# RAG jurídico — consecuencias del incumplimiento de un contrato

Proyecto final SI3003 · Opción A (sistema RAG) · **LangChain + NVIDIA NIM + Qdrant**



## 1. Repositorio y dependencias

En Colab, escribe la URL del repositorio del equipo. En local, abre el notebook desde la carpeta `notebooks/` del proyecto y esta celda solo ubicará la raíz.

In [1]:
import os, sys
from pathlib import Path

URL_REPO = ""  # p. ej. "https://github.com/<usuario>/<repositorio>.git" (solo necesario en Colab)

if not (Path.cwd() / "rag_juridico").is_dir():
    if (Path.cwd().parent / "rag_juridico").is_dir():      # ejecutando desde notebooks/
        os.chdir(Path.cwd().parent)
    elif URL_REPO:                                          # Colab
        destino = Path(URL_REPO.rstrip("/").split("/")[-1].removesuffix(".git"))
        if not destino.exists():
            !git clone -q {URL_REPO}
        os.chdir(destino)
    else:
        raise RuntimeError("Indica URL_REPO o abre el notebook desde la carpeta del proyecto.")

sys.path.insert(0, str(Path.cwd()))
print("Raíz del proyecto:", Path.cwd())

Raíz del proyecto: c:\Users\Lenovo\OneDrive\Escritorio\I.M\Inteligencia Artificial\Proyecto\RAG_Juridico


In [ ]:
%pip install -q -r requirements.txt

## 2. Clave de NVIDIA (privada)



In [2]:
from rag_juridico.config import Config
from rag_juridico.proveedores import obtener_api_key

obtener_api_key()
config = Config.desde_entorno()
print("✅ Clave configurada")
print("Embeddings:", config.modelo_embeddings, "| LLM:", config.modelo_generacion)

✅ Clave configurada
Embeddings: nvidia/nemotron-3-embed-1b | LLM: nvidia/nemotron-3.5-lightning-30b-a3b


## 3. Ingesta

Lee los documentos de `data/leyes` y `data/sentencias`, los fragmenta (las leyes **por artículo**), genera embeddings y los guarda en Qdrant (`storage/qdrant`). Solo hay que repetirla cuando cambian los documentos.

In [8]:
from rag_juridico.ingesta import ingestar

manifiesto = ingestar(config)

1/4 Cargando documentos de C:\Users\Lenovo\OneDrive\Escritorio\I.M\Inteligencia Artificial\Proyecto\RAG_Juridico\data ...
  [ley      ] Código Civil colombiano  (618 pág., 1,518,597 caracteres)
  [ley      ] Código de Comercio colombiano - Decreto 410 de 1971  (635 pág., 1,347,497 caracteres)
  [sentencia] Responsabilidad civil contractual  (30 pág., 71,886 caracteres)
  [sentencia] Tribunal Superior de Medellín - Responsabilidad civil contractual  (58 pág., 98,797 caracteres)
  ⚠ sentencias/Responsabilidad_Civil_Contractual.pdf: sin origen declarado en data/fuentes.json.
2/4 Fragmentando ...
  4 documentos -> 5642 fragmentos {'ley': 5460, 'sentencia': 182}
3/4 Generando embeddings (nvidia/nemotron-3-embed-1b) ...
  5642/5642
4/4 Indexando en qdrant (dimensión 2048) ...
✅ Índice listo: 5642 fragmentos de 4 documentos.


In [3]:
# Inspección: así queda un fragmento con sus metadatos
from rag_juridico.ingesta import leer_fragmentos

fragmentos = leer_fragmentos(config.ruta_fragmentos)
ejemplo = fragmentos[len(fragmentos) // 2]
print(ejemplo.encabezado())
print("-" * 80)
print(ejemplo.texto[:600])

Ley | Código Civil colombiano | art. 2466, p. 542
--------------------------------------------------------------------------------
ARTICULO 2466. <COMPENSACION DE FRUTO S CON LOS INTERESES>. Las partes podrán
estipular que los frutos se compensen con los intereses, en su totalidad, o hasta concurrencia de valores.

Los intereses que estipularen estarán sujetos, en caso de lesión enorme, a la misma reducción que en el
caso de mutuo.


## 4. Consultar un caso

`responder()` ejecuta cinco etapas: (1) el LLM identifica los elementos del caso y redacta consultas en lenguaje jurídico; (2) recuperación híbrida (similitud coseno + BM25, fusionadas con RRF) con cupo por tipo de fuente; (3) control de suficiencia; (4) generación restringida al contexto con citas `[F#]`; (5) verificación de las citas.

In [3]:
from IPython.display import Markdown, display
from rag_juridico.pipeline import AsistenteJuridico
from rag_juridico.presentacion import a_markdown

asistente = AsistenteJuridico.desde_storage(config)

c:\Users\Lenovo\anaconda3\lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:269: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


In [4]:
caso = (
    "¿Qué establece el Código Civil sobre la entrega de una cosa vendida?"
)
respuesta = asistente.responder(caso)
display(Markdown(a_markdown(respuesta)))

**Estado:** 🟡 Respondida parcialmente (el contexto cubre solo una parte)

### Hechos relevantes
El usuario consulta sobre lo que establece el Código Civil colombiano respecto a la obligación de entrega en un contrato de compraventa. No se han expuesto hechos concretos de un caso particular, sino la pregunta doctrinal general sobre la entrega de la cosa vendida.

### Qué dicen las fuentes
- **[F3]** Define la compraventa como el contrato en que una parte se obliga a dar una cosa y la otra a pagarla en dinero.
- **[F1] (Art. 1880)** Establece que las obligaciones del vendedor se reducen a dos: la entrega o tradición, y el saneamiento de la cosa vendida. La tradición se sujetará a las reglas del título VI del libro II.
- **[F5] (Art. 1884)** Reitera que el vendedor está obligado a entregar lo que reza el contrato.
- **[F2] (Art. 1882)** Regula el tiempo de entrega: el vendedor debe entregar inmediatamente después del contrato o a la época prefijada. Si hay retardo, el comprador puede perseverar en el contrato o desistir y en ambos casos tener derecho a indemnización de perjuicios. Si la fortuna del comprador mengua considerablemente después del contrato, no se podrá exigir la entrega si no se paga o asegura el pago del precio.
- **[F4] (Art. 928 del Código de Comercio)** Establece que el vendedor está obligado a entregar lo que reza el contrato, con todos sus accesorios, en las mismas condiciones que tenía al momento de perfeccionarse. Si la cosa es un cuerpo cierto, deberá conservarla hasta su entrega bajo pena de indemnizar los perjuicios, salvo que la pérdida o deterioro se deban a fuerza mayor o caso fortuito (carga de la prueba recae en el vendedor).
- **[F7] (Art. 924 del Código de Comercio)** Fija el plazo de entrega: dentro del plazo estipulado; a falta de estipulación, dentro de las veinticuatro horas siguientes al perfeccionamiento del contrato, salvo que de la naturaleza de la cosa o de la forma de entrega se derive la necesidad de un plazo mayor.

### Alcance y límites
Los fragmentos recuperados permiten identificar las reglas generales sobre la obligación de entrega en la compraventa según el Código Civil y el Código de Comercio colombianos. No obstante, para aplicar estas normas a un caso concreto sería necesario conocer la voluntad de las partes (plazo pactado o no), la calidad de la cosa y si existen circunstancias que afecten la exigibilidad de la entrega (p.ej., retroceso de la fortuna del comprador, fuerza mayor). Las fuentes no resuelven conflictos entre regímenes ni interpretan situaciones fácticas específicas.

**⚠ Advertencias de verificación**
- El modelo no declaró la suficiencia. La respuesta se clasificó provisionalmente como parcial.

---
### Elementos identificados en el caso
- Tipo de contrato: compraventa
- Partes: vendedor (obligado a entregar); comprador (derecho a recibir)
- Obligaciones: entregar la cosa vendida
- Incumplimiento: no determinado
- Causa: no_determinada
- Cláusulas relevantes: obligación de entrega
- Pregunta jurídica: Qué establece el Código Civil sobre la obligación de entrega en el contrato de compraventa
- Datos que faltan: voluntad de las partes, calidad de la cosa, plazo pactado o notificación de entrega

### Fuentes recuperadas
**[F1] Ley · Código Civil colombiano · art. 1880, p. 436** — citada, similitud 0.65
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf
  > ARTICULO 1880. <OBLIGACIONES DEL VENDEDOR>. Las obligaciones del vendedor se reducen
  > en general a dos: la entrega o tradición, y el saneamiento de la cosa vendida.
  > 
  > La tradición se sujetará a las reglas dadas en el título VI del libro II.
  > 
  > CODIGO CIVIL COLOMBIANO
  > Página 437 de 618

**[F2] Ley · Código Civil colombiano · art. 1882, p. 437** — citada, similitud 0.63
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf
  > ARTICULO 1882. <TIEMPO DE ENTREGA Y RETARDO>. El vendedor es obligado a entregar la
  > cosa vendida inmediatamente después del contrato, o a la época prefijada en él.
  > 
  > Si el vendedor, por hecho o culpa suya ha retardado la entrega, podrá el comprador, a su arbitrio,
  > perseverar en el contrato o desistir de é l y en ambos casos con derecho para ser indemnizado de los
  > perjuicios según las reglas generales.
  > 
  > Todo lo cual se entiende si el comprador ha pagado o está pronto a pagar el precio íntegro o ha
  > estipulado pagar a plazo.
  > 
  > Pero si después del contrato hubie re menguado considerablemente la fortuna del comprador, de modo
  > que el vendedor se halle en peligro inminente de perder el precio, no se …

**[F3] Ley · Código Civil colombiano · art. 1849, p. 431** — citada, similitud 0.63
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf
  > ARTICULO 1849. <CONCEPTO DE COMPRAVENTA>. La compraventa es un contrato en que una
  > de las partes se obliga a dar una cosa y la otra a pa garla en dinero. Aquélla se dice vender y ésta
  > comprar. El dinero que el comprador da por la cosa vendida se llama precio.

**[F4] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 928, p. 219** — citada, similitud 0.64
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6
  > Art. 928._Obligación de entregar. El vende dor estará obligado a entregar lo que reza el
  > contrato, con todos sus accesorios, en las mismas condiciones que tenía al momento de
  > perfeccionarse; y si la cosa vendida es un cuerpo cierto, estará obligado a conservarla
  > hasta su entrega so pena de indemniz ar los perjuicios al comprador, salvo que la pérdida
  > o deterioro se deban a fuerza mayor o caso fortuito, cuya prueba corresponderá al
  > vendedor.
  > Conc.: 923, 927, 931; C. Civil 1605 a 1608, 1857, 1884.

**[F5] Ley · Código Civil colombiano · art. 1884, p. 437** — citada, similitud 0.67
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf
  > ARTICULO 1884. <OBJETO DE LA ENTREGA>. El vendedor es obligado a entregar lo que reza el
  > contrato.

**[F6] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 909, p. 215** — no citada, coincidencia léxica
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6
  > Art. 909._Gastos del contrato de compraventa. Los gastos que ocasione la celebración
  > del contrato se dividirán por partes iguales entre los contratantes, si éstos no acuerdan
  > otra cosa.
  > Salvo costumbre comercial o pacto en contrario, los gasto s de entrega de la cosa vendida
  > corresponderán al vendedor y los de recibo de la misma, al comprador.
  > Conc.: 3o., 871, 923; C. Civil 15, 1862, 1881.

**[F7] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 924, p. 219** — citada, similitud 0.62
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6
  > Art. 924._Plazo para la entrega. El vendedor deberá hacer la entrega de la cosa dentro
  > del plazo estipulado. A falta de estipulación deberá entregarla dentro de las veinticuatro
  > horas siguientes al perfeccionamiento del contrato, salvo que d e la naturaleza del mismo
  > o de la forma como deba hacerse la entrega se desprenda que para verificarla se requiere
  > un plazo mayor.
  > Conc.: 870, 912, 943; C. Civil, 1880, 1882; C. de P. C. 121.

**[F8] Sentencia · Tribunal Superior de Medellín - Responsabilidad civil contractual · Consideraciones, p. 46** — no citada, coincidencia léxica
  - Archivo: `sentencias/Sentencia_2022.pdf`
  - Origen: https://tribunalmedellin.com/images/decisiones/civil/2023/05001310301020200008501.pdf
  > (iv) Entre el señor Correcha Jiménez y Luz Esther
  > Correcha de Rodríguez no existió pacto preparatorio alguno,
  > o al menos el mismo no se aportó al plenario ; pero sí lo
  > 
  > 24 F. 27, cdno. 3 y 378, cdno. 1, en ese orden.
  > 25 Ese monto se depositó en la cuenta del promitente vendedor (f. 295, íd.).
  > 26 F. 475, íd.
  > 
  > Radicación n.° 73001 -31-03-006-2011 -00139-01
  > 47 hubo entre aquella y la convocada Díaz Cabrera , respecto
  > del inmueble cuya transferencia constituye el objeto del
  > Contratos 5. En esa promesa de compraventa (también con
  > escasas jornadas de diferencia) , se pactó:

_Este prototipo académico identifica y explica información de documentos indexados; no determina responsabilidad contractual ni sustituye asesoría jurídica profesional._

## 5. Los cinco casos de prueba

Ejecuta `evaluacion/casos_prueba.json` y genera un informe en `evaluacion/resultados/` con pregunta, resultado esperado, respuesta, fuentes y un espacio para la **valoración manual** del equipo.

In [4]:
from rag_juridico.evaluacion import evaluar

informe = evaluar(asistente)
display(Markdown(informe.read_text(encoding="utf-8")))

▶ C1 (Consulta documental — compraventa y entrega) ...
▶ C2 (Consulta documental — arrendamiento y cláusula penal) ...
▶ C3 (Consulta documental — contrato de obra) ...
▶ C4 (Comparación documental — pérdida fortuita de cuerpo cierto) ...
▶ C5 (Fuera de alcance — indemnización laboral) ...
✅ Informe: C:\Users\Lenovo\OneDrive\Escritorio\I.M\Inteligencia Artificial\Proyecto\RAG_Juridico\evaluacion\resultados\resultados_20261114_232824.md


# Resultados de evaluación — 20261114_232824

## C1 — Consulta documental — compraventa y entrega

**Pregunta / caso:** Ana compró un apartamento a Luis, pagó todo el precio y la fecha pactada para la entrega pasó hace tres meses sin que Luis entregara el inmueble. Según los documentos disponibles, ¿qué establece el Código Civil sobre la obligación de entrega del vendedor y sobre las alternativas que menciona cuando hay retraso? No determines si Luis debe indemnizar a Ana en este caso concreto.

**Resultado esperado:** Debe identificar la falta de entrega y explicar fielmente el artículo 1882 del Código Civil: obligación de entregar en el momento pactado y alternativas de perseverar o desistir cuando el retraso sea por hecho o culpa del vendedor, con referencia a perjuicios según esa condición. No debe presumir culpa ni decidir una indemnización específica.

**Estado obtenido:** 🟡 Respondida parcialmente (el contexto cubre solo una parte) (mejor similitud 0.75, 8.0 s)

**Respuesta obtenida:**

### Hechos relevantes
Ana compró un apartamento a Luis, pagó el precio total y la fecha pactada para la entrega venció hace tres meses sin que Luis entregara el inmueble.

### Qué dicen las fuentes
Según el Código Civil colombiano, el vendedor tiene la obligación de entregar la cosa vendida inmediatamente después del contrato o a la época prefijada [F1, art. 1882]. Si el vendedor retrasa la entrega por su hecho o culpa, el comprador tiene dos alternativas a su arbitrio: perseverar en el contrato o desistir de él. En ambos casos, el comprador tiene derecho a ser indemnizado de los perjuicios según las reglas generales [F1, art. 1882]. Además, el vendedor está obligado a entregar lo que reza el contrato [F3, art. 1884].

### Alcance y límites
Los fragmentos recuperados establecen que el retraso en la entrega otorga al comprador la opción de mantener la obligación o rescindir el contrato, más el derecho a indemnización de perjuicios. No se determina en el caso concreto si el retraso fue por hecho o culpa del vendedor, ni si el comprador ejerció alguna de las alternativas legales. Tampoco se identifica si existen cláusulas pactadas que modifiquen estos efectos. Para una decisión sobre la situación concreta se requeriría información sobre la conducta de las partes y eventuales acuerdos específicos.

**Fuentes recuperadas:**

**[F1] Ley · Código Civil colombiano · art. 1882, p. 437** — citada, similitud 0.75
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F2] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 943, p. 222** — no citada, similitud 0.62
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6

**[F3] Ley · Código Civil colombiano · art. 1884, p. 437** — citada, similitud 0.70
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F4] Ley · Código Civil colombiano · art. 1880, p. 436** — no citada, similitud 0.70
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F5] Ley · Código Civil colombiano · art. 1984, p. 452** — no citada, similitud 0.60
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F6] Ley · Código Civil colombiano · art. 667, p. 210** — no citada, similitud 0.57
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F7] Ley · Código Civil colombiano · art. 1930, p. 443** — no citada, similitud 0.63
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F8] Sentencia · Tribunal Superior de Medellín - Responsabilidad civil contractual · Antecedentes, p. 10** — no citada, coincidencia léxica
  - Archivo: `sentencias/Sentencia_2022.pdf`
  - Origen: https://tribunalmedellin.com/images/decisiones/civil/2023/05001310301020200008501.pdf

**Comprobaciones automáticas (ayuda, no reemplazan la revisión manual):**
- ¿Estado esperado? sí (esperado: respondida, parcial)
- Tipos de fuente esperados que faltaron: ninguno
- Términos esperados ausentes: ninguno
- Advertencias de citas: El modelo no declaró la suficiencia. La respuesta se clasificó provisionalmente como parcial.

**Valoración manual del equipo:** _(¿es relevante?, ¿cada afirmación está respaldada por el fragmento citado?, ¿coincide con el documento original?)_

---

## C2 — Consulta documental — arrendamiento y cláusula penal

**Pregunta / caso:** En un arrendamiento de local comercial, el arrendatario dejó de pagar tres cánones y el contrato contempla una cláusula penal de dos cánones. ¿Qué dicen los fragmentos recuperados sobre la falta de pago de la renta y sobre la cláusula penal? Identifica los artículos y sus condiciones, sin calcular cuánto se puede cobrar ni afirmar que la penalidad es válida en este contrato.

**Resultado esperado:** Debe describir, si se recuperan, el artículo 2035 del Código Civil sobre la falta de pago de renta y las condiciones que menciona, y el artículo 867 del Código de Comercio sobre cláusula penal. Debe distinguir entre describir esas reglas y decidir su aplicación. No debe afirmar que se pueden cobrar exactamente cinco cánones ni asegurar la validez o acumulación de la pena.

**Estado obtenido:** 🟡 Respondida parcialmente (el contexto cubre solo una parte) (mejor similitud 0.52, 11.4 s)

**Respuesta obtenida:**

### Hechos relevantes
El arrendatario de un local comercial dejó de pagar tres cánones (períodos de renta) y el contrato incluye una cláusula penal estipulada en dos cánones.

### Qué dicen las fuentes

**Sobre la falta de pago de la renta:**
- **[F3] Art. 2035**: Establece que la mora de un período entero en el pago de la renta, después de dos reconvenciones (mínimo cuatro días entre ellas), da derecho al arrendador a cesar el arrendamiento si no se presta seguridad de pago dentro de un plazo razonable (mínimo treinta días).
- **[F7] Art. 2013**: Señala que, si el arrendamiento cesa por desahucio o vencimiento de duración, el arrendatario debe pagar la renta de todos los días que faltan para que cese, aunque haya restituido voluntariamente el local antes del último día.
- **[F6] Art. 2003**: Indica que, cuando el arrendamiento se termina por culpa del arrendatario, este debe indemnizar perjuicios y pagar la renta por el tiempo que faltaba hasta que, desahucio mediante, el arrendador pudiera cesar el contrato. El arrendatario puede eximitarse proponiendo un sustituto idóneo y prestando fianza.

**Sobre la cláusula penal:**
- **[F1] Art. 949 (Código de Comercio)** y **[F2] Art. 1592 (Código Civil)**: Definen la cláusula penal como aquel pacto por el cual una persona se sujeta a una pena (entregar o hacer algo) en caso de no ejecutar o retardar la obligación principal. El Art. 949 conecta esta figura con el Art. 867 del Código Civil.
- **[F5] Art. 1593**: Establece que la nulidad de la obligación principal acarrea la de la cláusula penal, pero la nulidad de la penal no extingue la obligación principal. Además, la pena sigue siendo exigible aunque la obligación principal carezca de efecto por falta de consentimiento de la persona prometida, o cuando la estipulación es a favor de un tercero.

### Alcance y límites
Los fragmentos recuperados permiten identificar las reglas sobre la mora en el pago de la renta (requisitos para cesar el arrendamiento) y la definición y efectos de la cláusula penal (vinculación a la obligación principal y consecuencias de su nulidad). No obstante, no se encuentran en el corpus disposiciones que regulen específicamente los límites, la procedencia o el cálculo de la penalidad estipulada en cánones en contratos de arrendamiento comercial, ni se definen las condiciones particulares de aplicación de la cláusula penal de dos cánones en el caso de falta de pago de tres cánones. Para un análisis sobre la validez o el monto de la penalidad en este contrato concreto, se requeriría consultar el texto completo del contrato y, potencialmente, normas adicionales no recuperadas en este contexto.

**Fuentes recuperadas:**

**[F1] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 949, p. 223** — citada, similitud 0.48
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6

**[F2] Ley · Código Civil colombiano · art. 1592, p. 377** — citada, similitud 0.45
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F3] Ley · Código Civil colombiano · art. 2035, p. 461** — citada, similitud 0.43
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F4] Ley · Código Civil colombiano · art. 2486, p. 544** — no citada, similitud 0.46
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F5] Ley · Código Civil colombiano · art. 1593, p. 377** — citada, similitud 0.40
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F6] Ley · Código Civil colombiano · art. 2003, p. 455** — citada, similitud 0.39
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F7] Ley · Código Civil colombiano · art. 2013, p. 457** — citada, similitud 0.40
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F8] Sentencia · Tribunal Superior de Medellín - Responsabilidad civil contractual · Consideraciones, p. 22** — no citada, coincidencia léxica
  - Archivo: `sentencias/Sentencia_2022.pdf`
  - Origen: https://tribunalmedellin.com/images/decisiones/civil/2023/05001310301020200008501.pdf

**Comprobaciones automáticas (ayuda, no reemplazan la revisión manual):**
- ¿Estado esperado? sí (esperado: respondida, parcial)
- Tipos de fuente esperados que faltaron: ninguno
- Términos esperados ausentes: ninguno
- Advertencias de citas: El modelo no declaró la suficiencia. La respuesta se clasificó provisionalmente como parcial.

**Valoración manual del equipo:** _(¿es relevante?, ¿cada afirmación está respaldada por el fragmento citado?, ¿coincide con el documento original?)_

---

## C3 — Consulta documental — contrato de obra

**Pregunta / caso:** Una empresa contrató una remodelación y debía pagar un anticipo del 30 % antes de iniciar. No pagó el anticipo y el contratista no empezó la obra. ¿Qué disposiciones recuperadas tratan el incumplimiento y los perjuicios en los contratos de obra? Explica lo que dicen, pero no decidas si una demanda de alguna de las partes prosperaría.

**Resultado esperado:** Debe reconocer los hechos del anticipo y explicar el artículo 2056 del Código Civil, si está recuperado, sobre la obra material y la reclamación de perjuicios según las reglas generales. No debe convertir la facultad de quien encarga la obra para hacerla cesar en un supuesto derecho del contratista a suspenderla. Si no se recupera una norma sobre obligaciones recíprocas, debe decirlo sin inventar esa regla.

**Estado obtenido:** 🟡 Respondida parcialmente (el contexto cubre solo una parte) (mejor similitud 0.59, 11.4 s)

**Respuesta obtenida:**

### Hechos relevantes
Una empresa (comitente) contrató una remodelación y acordó pagar un anticipo del 30 % antes de iniciar la obra. La empresa no realizó ese pago y, como consecuencia, el contratista no comenzó los trabajos.

### Qué dicen las fuentes
- **[F6]** Art. 1552: "Lo que se paga antes de cumplirse el plazo, no está sujeto a restitución. Esta regla no se aplica a los plazos que tienen el valor de condiciones." Este fragmento establece la regla general sobre pagos anticipados, indicando que, salvo que el plazo tenga valor de condición, el pago anticipado no genera derecho a devolución.

- **[F2]** Art. 2056: "Habrá lugar a reclamación de perjuicios, según las reglas generales de los contratos, siempre que por una o por otra parte no se ha ya ejecutado lo convenido, o se haya retardado su ejecución." Este artículo permite reclamar perjuicios cuando no se ejecuta lo convenido. Además, señala que "el que encargó la obra, aún en el caso de haberse estipulado un precio único y total por ella, podrá hacerla cesar, reembolsando al artífice todos los costos, y dándole lo que valga el trabajo hecho, y lo que hubiera podido ganar en la obra". Esto implica que, si el comitente no paga el anticipo y el contratista no inicia, el contratista podría cesar en la ejecución y reclamar por el trabajo no realizado y los lucros cesantes.

- **[F5]** Art. 1616: "Si no se puede imputar dolo al deudor, solo es responsable de los perjuicios que se previeron o pudieron preverse al tiempo del contrato; pero si hay dolo, es responsable de todos los perjuicios que fueron consecuencia inmediata o directa de no haberse cumplido la obligación o de haberse demorado su cumplimiento. La mora producida por fuerza mayor o caso fortuito, no da lugar a indemnización de perjuicios." Este fragmento delimita la responsabilidad por perjuicios: solo se indemnizan los previstos o previsibles, salvo que haya dolo. La mora por fuerza mayor o caso fortuito exime de indemnización.

- **[F7]** Art. 1614: Define "daño emergente" (perjuicio o pérdida por no cumplimiento) y "lucro cesante" (ganancia que se deja de reportar). Estas figuras son la base para calcular los perjuicios en caso de incumplimiento.

- **[F1]** Art. 2062: Trata la resolución del contrato por muerte del artífice o empresario, y la obligación de recibir trabajos o materiales preparados. No es directamente aplicable al caso de no pago de anticipo, pero forma parte del capítulo sobre contratos de construcción.

### Alcance y límites
Los fragmentos recuperados permiten explicar las reglas generales sobre pagos anticipados ([F6]), la facultad de cesar la obra y reclamar perjuicios por falta de ejecución ([F2]), y los conceptos de daño emergente y lucro cesante ([F7]), así como el límite de responsabilidad por perjuicios previstos frente al dolo ([F5]). No se dispone en el corpus de una norma específica que regule el anticipo del 30 % ni de una cláusula penal o resolutoria pactada por las partes. Para determinar el régimen exacto aplicable a este caso (si el anticipo es exigible, si su falta permite la resolución del contrato y qué perjuicios se pueden reclamar), sería necesario conocer el contenido exacto de la cláusula de anticipo y si las partes estipularon penalidades o condiciones resolutorias.

**Fuentes recuperadas:**

**[F1] Ley · Código Civil colombiano · art. 2062, p. 467** — citada, similitud 0.51
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F2] Ley · Código Civil colombiano · art. 2056, p. 465** — citada, similitud 0.59
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F3] Ley · Código Civil colombiano · art. 2059, p. 466** — no citada, similitud 0.57
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F4] Ley · Código Civil colombiano · art. 2052, p. 465** — no citada, coincidencia léxica
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F5] Ley · Código Civil colombiano · art. 1616, p. 382** — citada, similitud 0.52
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F6] Ley · Código Civil colombiano · art. 1552, p. 371** — citada, similitud 0.50
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F7] Ley · Código Civil colombiano · art. 1614, p. 382** — citada, similitud 0.48
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F8] Sentencia · Responsabilidad civil contractual · p. 5** — no citada, coincidencia léxica
  - Archivo: `sentencias/Responsabilidad_Civil_Contractual.pdf`

**Comprobaciones automáticas (ayuda, no reemplazan la revisión manual):**
- ¿Estado esperado? sí (esperado: respondida, parcial)
- Tipos de fuente esperados que faltaron: ninguno
- Términos esperados ausentes: ninguno
- Advertencias de citas: El modelo no declaró la suficiencia. La respuesta se clasificó provisionalmente como parcial.

**Valoración manual del equipo:** _(¿es relevante?, ¿cada afirmación está respaldada por el fragmento citado?, ¿coincide con el documento original?)_

---

## C4 — Comparación documental — pérdida fortuita de cuerpo cierto

**Pregunta / caso:** Un caballo individualmente determinado fue vendido y murió por un rayo antes de su entrega, sin culpa ni mora del vendedor. El comprador ya había pagado. ¿Qué diferencias aparecen entre el artículo 1876 del Código Civil y el artículo 929 del Código de Comercio sobre quién soporta el riesgo? ¿Qué dice el artículo 1729 sobre la extinción de la obligación? Limítate a comparar el texto recuperado; no decidas qué régimen aplica ni si debe devolverse el precio.

**Resultado esperado:** Debe contrastar expresamente el artículo 1876 del Código Civil (riesgo a cargo del comprador como regla descrita) con el 929 del Código de Comercio (riesgo a cargo del vendedor antes de la entrega, con excepciones). Si se recupera el artículo 1729, debe describir la extinción de la obligación de entregar por pérdida del cuerpo cierto y sus salvedades. No debe concluir definitivamente sobre la devolución del precio, la mora del comprador o el régimen aplicable.

**Estado obtenido:** 🟡 Respondida parcialmente (el contexto cubre solo una parte) (mejor similitud 0.63, 9.2 s)

**Respuesta obtenida:**

### Hechos relevantes
Se consulta sobre un caso de compraventa de un caballo individualmente determinado que murió por rayo antes de la entrega, tras haber sido pagado el precio por el comprador. La pregunta se centra en comparar el artículo 1876 del Código Civil, el artículo 929 del Código de Comercio y el artículo 1729 del Código Civil.

### Qué dicen las fuentes

**Artículo 1876 del Código Civil [F2]:** Establece que la pérdida, deterioro o mejora de la cosa cierta pertenece al comprador desde la perfección del contrato, aunque no se haya entregado la cosa. La excepción es cuando la venta es bajo condición suspensiva: si perece la cosa mientras pende la condición, la pérdida recae en el vendedor.

**Artículo 929 del Código de Comercio [F1]:** En la venta de un "cuerpo cierto", el riesgo de pérdida por fuerza mayor o caso fortuito antes de la entrega corresponde al vendedor, salvo que el comprador se constituya en mora de recibirlo y que la fuerza mayor o el caso fortuito no lo hubiera destruido sin la mora del comprador. En ese caso, el comprador debe el precio íntegro.

**Artículo 1729 del Código Civil [F3]:** Cuando el cuerpo cierto que se debe perece (por destrucción, al dejar de estar en el comercio o al desaparecer), se extingue la obligación, salvo las excepciones de los artículos subsiguientes.

### Alcance y límites
- El artículo 1876 (Código Civil) sitúa el riesgo en el comprador desde la perfección del contrato, sin distinguir expresamente la causa de la pérdida (fuerza mayor o caso fortuito) en el fragmento recuperado.
- El artículo 929 (Código de Comercio) atribuye el riesgo al vendedor antes de la entrega, pero con la salvedad de que, si el comprador está en mora y la cosa se destruye sin que ello dependa de su mora, el comprador debe el precio.
- El artículo 1729 (Código Civil) regla la extinción de la obligación por pérdida de la cosa, pero menciona que salvan las excepciones de los artículos subsiguientes, lo que indica que la regla no es absoluta.
- No se determina el régimen aplicable al caso concreto, ya que el análisis requiere contrastar si el contrato se rige por el Código Civil o el Código de Comercio, y si la condición suspensiva o la mora del comprador están presentes.

**Fuentes recuperadas:**

**[F1] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 929, p. 219** — citada, similitud 0.63
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6

**[F2] Ley · Código Civil colombiano · art. 1876, p. 436** — citada, similitud 0.63
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F3] Ley · Código Civil colombiano · art. 1729, p. 402** — citada, similitud 0.61
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F4] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 2038, p. 572** — no citada, similitud 0.48
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6

**[F5] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 952, p. 224** — no citada, similitud 0.41
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6

**[F6] Ley · Código de Comercio colombiano - Decreto 410 de 1971 · art. 928, p. 219** — no citada, similitud 0.45
  - Archivo: `leyes/codigo_comercio.pdf`
  - Origen: https://www.supersociedades.gov.co/documents/107391/831466/Decreto%2B410%2Bdel%2B27%2Bde%2Bmarzo%2Bde%2B1971.pdf/16152a0c-ab79-5e92-7e13-36326dabfdd6

**[F7] Ley · Código Civil colombiano · art. 1899, p. 439** — no citada, coincidencia léxica
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**[F8] Ley · Código Civil colombiano · art. 1607, p. 380** — no citada, similitud 0.43
  - Archivo: `leyes/codigo_civil_colombia.pdf`
  - Origen: https://www.oas.org/dil/esp/codigo_civil_colombia.pdf

**Comprobaciones automáticas (ayuda, no reemplazan la revisión manual):**
- ¿Estado esperado? sí (esperado: respondida, parcial)
- Tipos de fuente esperados que faltaron: ninguno
- Términos esperados ausentes: ninguno
- Advertencias de citas: ninguna

**Valoración manual del equipo:** _(¿es relevante?, ¿cada afirmación está respaldada por el fragmento citado?, ¿coincide con el documento original?)_

---

## C5 — Fuera de alcance — indemnización laboral

**Pregunta / caso:** Trabajo hace cinco años con contrato laboral a término indefinido y mi empleador me despidió sin justa causa. ¿Cuántos días de salario me corresponden como indemnización por despido?

**Resultado esperado:** Debe clasificar la consulta como fuera del alcance documental y abstenerse brevemente de calcular días o citar reglas laborales no recuperadas. No debe dar fórmulas, cifras, artículos del Código Sustantivo del Trabajo no aportados por el corpus ni conclusiones sobre el derecho del usuario.

**Estado obtenido:** 🔴 El modelo declaró que el contexto no es suficiente (mejor similitud 0.00, 2.8 s)

**Respuesta obtenida:**

Esta consulta corresponde al derecho laboral, una materia fuera del alcance de este prototipo de consulta documental sobre contratos civiles y comerciales. No es posible responderla adecuadamente con el corpus disponible.

**Fuentes recuperadas:**

_No se recuperaron fragmentos suficientemente relacionados._

**Comprobaciones automáticas (ayuda, no reemplazan la revisión manual):**
- ¿Estado esperado? sí (esperado: sin_contexto, insuficiente)
- Tipos de fuente esperados que faltaron: ninguno
- Términos esperados ausentes: ninguno
- Advertencias de citas: ninguna

**Valoración manual del equipo:** _(¿es relevante?, ¿cada afirmación está respaldada por el fragmento citado?, ¿coincide con el documento original?)_

---
